<a href="https://colab.research.google.com/github/Emily-als/worklens-llm/blob/feature%2Fsynthetic-data-generation/04_synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
## 1. Environment Setup
# Verify GPU availability before loading the generation model.

!nvidia-smi

Mon Nov 23 11:22:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
## 2. Check Python, PyTorch and CUDA versions

import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU is required"

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

RuntimeError: THPDtypeType.tp_dict == nullptr INTERNAL ASSERT FAILED at "/pytorch/torch/csrc/Dtype.cpp":185, please report a bug to PyTorch. 

In [ ]:
## 3. Recreate Python Environments
# Restore the isolated environments after a Colab runtime reset.

# Install uv
%pip install -q uv

# Install Python 3.11
!uv python install 3.11

# Create isolated environments
!uv venv --python 3.11 --seed /content/worklens_env
!uv venv --python 3.11 --seed /content/vllm_env

# Verify both environments
from pathlib import Path

assert Path("/content/worklens_env/bin/python").exists()
assert Path("/content/vllm_env/bin/python").exists()

print("Both Python environments recreated successfully.")

In [5]:
## 4. Install DataDreamer
# Install compatible package versions in the isolated environment.
# Pin versions to avoid dependency conflicts after runtime resets.

!/content/worklens_env/bin/python -m pip install \
    "datadreamer.dev==0.46.0" \
    "datasets==3.2.0" \
    "pyarrow==17.0.0" \
    "numpy==1.26.4" \
    > /content/datadreamer_install.log 2>&1

# Check installation status and dependencies.
!tail -n 5 /content/datadreamer_install.log
!/content/worklens_env/bin/python -m pip check
!tail -n 10 /content/datadreamer_install.log

  Created wheel for ring: filename=ring-0.10.1-py3-none-any.whl size=36383 sha256=fce05cb225fe2ee8768a4ccb26cc14659aef33e5eb7fe335f8a32cc79e1eefaf
  Stored in directory: /root/.cache/pip/wheels/f9/38/2a/39544379dcfd57247257c771c6d424af7fbd39e496cf48d73e
  Created wheel for sqlitedict: filename=sqlitedict-2.1.0-py3-none-any.whl size=16957 sha256=3a53b50099c9b047b9a4e3d347487b5cf51c2bd304d07402db5d5ec2a1d7f0dc
  Stored in directory: /root/.cache/pip/wheels/73/63/89/7210274f9b7fb033b8f22671f64c0e0b55083d30c3c046a3ff
Successfully built ring sqlitedict



In [6]:
## 5. Restore vLLM

!/content/vllm_env/bin/python -m pip install \
    "vllm==0.34.0" \
    > /content/vllm_install.log 2>&1

!tail -n 10 /content/vllm_install.log

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 50.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 34.3 MB/s  0:00:00
ERROR: Operation cancelled by user



In [7]:
## 6. Verify Installations

!/content/worklens_env/bin/python -c \
    "import datadreamer; print('DataDreamer OK')"

!/content/vllm_env/bin/python -c \
    "import vllm; print('vLLM:', vllm.__version__)"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/worklens_env/lib/python3.11/site-packages/datadreamer/__init__.py", line 64, in <module>
    from .datadreamer import DataDreamer
  File "/content/worklens_env/lib/python3.11/site-packages/datadreamer/datadreamer.py", line 18, in <module>
    from ._patches.datasets_reset_state_hack import (
  File "/content/worklens_env/lib/python3.11/site-packages/datadreamer/_patches/datasets_reset_state_hack.py", line 7, in <module>
    from datasets.iterable_dataset import (
ImportError: cannot import name 'TypedExamplesIterable' from 'datasets.iterable_dataset' (/content/worklens_env/lib/python3.11/site-packages/datasets/iterable_dataset.py)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'vllm'


In [1]:
## Check Current Installation Status
# Check whether the virtual environments and packages are available.

from pathlib import Path

for name in ["worklens_env", "vllm_env"]:
    python_path = Path(f"/content/{name}/bin/python")
    print(f"{name}: {'Exists' if python_path.exists() else 'Missing'}")

!test -f /content/datadreamer_install.log && tail -n 3 /content/datadreamer_install.log
!test -f /content/vllm_install.log && tail -n 3 /content/vllm_install.log

worklens_env: Exists
vllm_env: Exists

ERROR: Operation cancelled by user



In [2]:
## Check Current Package Status
# Check installed packages without loading the large ML libraries.

import subprocess

environments = {
    "DataDreamer": "/content/worklens_env/bin/python",
    "vLLM": "/content/vllm_env/bin/python"
}

for name, python_path in environments.items():
    package = "datadreamer.dev" if name == "DataDreamer" else "vllm"

    result = subprocess.run(
        [python_path, "-m", "pip", "show", package],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        version = next(
            (line for line in result.stdout.splitlines()
             if line.startswith("Version:")),
            "Version unknown"
        )
        print(f"{name}: Installed — {version}")
    else:
        print(f"{name}: Not fully installed")

DataDreamer: Installed — Version: 0.46.0
vLLM: Not fully installed


In [3]:
## 7. Complete vLLM Installation
# Resume installation in the existing vllm_env.
# Do not recreate the virtual environment.

!uv pip install \
    --python /content/vllm_env/bin/python \
    "vllm==0.34.0" \
    > /content/vllm_install.log 2>&1

# Show the final installation messages.
!tail -n 15 /content/vllm_install.log

 + opentelemetry-exporter-otlp==1.45.2
 + opentelemetry-exporter-otlp-common==0.66b2
 + opentelemetry-exporter-otlp-proto-grpc==1.45.2
 + opentelemetry-exporter-otlp-proto-http==1.45.2
 + opentelemetry-semantic-conventions-ai==0.5.1
 + quack-kernels==0.6.5
 + tilelang==0.1.12
 + tokenizers==0.23.3
 + tokenspeed-mla==2.0.3
 + torch==2.14.0
 + torch-c-dlpack-ext==0.1.5
 + torchvision==0.29.0
 + transformers==5.22.2
 + vllm==0.34.0
 + xgrammar==0.2.8


In [4]:
## 8. Verify Both Environments
# Check that DataDreamer and vLLM can be imported successfully.

import subprocess

checks = [
    (
        "DataDreamer",
        "/content/worklens_env/bin/python",
        "import datadreamer; print('DataDreamer: OK')"
    ),
    (
        "vLLM",
        "/content/vllm_env/bin/python",
        "import vllm; print('vLLM version:', vllm.__version__)"
    )
]

for name, python_path, code in checks:
    print(f"\nChecking {name}...", flush=True)

    result = subprocess.run(
        [python_path, "-c", code],
        capture_output=True,
        text=True,
        timeout=120
    )

    print(result.stdout)

    if result.returncode != 0:
        print("ERROR:")
        print(result.stderr[-3000:])


Checking DataDreamer...

ERROR:
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/worklens_env/lib/python3.11/site-packages/datadreamer/__init__.py", line 64, in <module>
    from .datadreamer import DataDreamer
  File "/content/worklens_env/lib/python3.11/site-packages/datadreamer/datadreamer.py", line 18, in <module>
    from ._patches.datasets_reset_state_hack import (
  File "/content/worklens_env/lib/python3.11/site-packages/datadreamer/_patches/datasets_reset_state_hack.py", line 7, in <module>
    from datasets.iterable_dataset import (
ImportError: cannot import name 'TypedExamplesIterable' from 'datasets.iterable_dataset' (/content/worklens_env/lib/python3.11/site-packages/datasets/iterable_dataset.py)


Checking vLLM...
vLLM version: 0.34.0



In [5]:
## 9. Fix Hugging Face Datasets Compatibility
# Downgrade datasets to a version compatible with DataDreamer.
# Keep PyArrow 17.0.0 and NumPy 1.26.4 unchanged.

!/content/worklens_env/bin/python -m pip install \
    "datasets==2.19.2" \
    "pyarrow==17.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 15.7 MB/s  0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 3.6.0
    Uninstalling datasets-3.6.0:
      Successfully uninstalled datasets-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [datasets]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datadreamer-dev 0.46.0 requires datasets<4.0.0,>=3.2.0, but you have datasets 2.19.2 which is incompatible.


In [6]:
## 9.2. Fix Datasets Version
# Install a datasets version within DataDreamer's requirements.
# Keep NumPy and PyArrow versions unchanged.

!/content/worklens_env/bin/python -m pip install \
    "datasets==3.2.0" \
    "pyarrow==17.0.0" \
    "numpy==1.26.4"

# Check dependency consistency.
!/content/worklens_env/bin/python -m pip check

  Attempting uninstall: datasets
    Found existing installation: datasets 2.19.2
    Uninstalling datasets-2.19.2:
      Successfully uninstalled datasets-2.19.2
No broken requirements found.


In [7]:
## 9.3. Verify DataDreamer
# Test whether the compatible datasets version fixes the import error.

!/content/worklens_env/bin/python -c "import datasets; print('Datasets:', datasets.__version__); from datadreamer import DataDreamer; print('DataDreamer: OK')"

Datasets: 3.2.0
/content/worklens_env/lib/python3.11/site-packages/peft/import_utils.py:52: FutureWarning: Support for Transformers < 5.6 is deprecated in PEFT and will be removed in the next release, v1.0. You have Transformers 4.49.0. Please upgrade to Transformers >= 5.6.
  warnings.warn(
DataDreamer: OK


In [9]:
## 10. Verify GPU Compatibility
# Check GPU architecture and available memory.
# Run this code in the isolated vLLM environment.

import subprocess

code = """
import torch
import vllm

gpu_id = 0

print("vLLM version:", vllm.__version__)
print("GPU:", torch.cuda.get_device_name(gpu_id))
print("Compute capability:", torch.cuda.get_device_capability(gpu_id))

total_memory = (
    torch.cuda.get_device_properties(gpu_id).total_memory
    / (1024 ** 3)
)

print(f"Total GPU memory: {total_memory:.2f} GiB")
print("CUDA available:", torch.cuda.is_available())
"""

result = subprocess.run(
    ["/content/vllm_env/bin/python", "-c", code],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("ERROR:")
    print(result.stderr[-3000:])

vLLM version: 0.34.0
GPU: Tesla T4
Compute capability: (7, 5)
Total GPU memory: 14.56 GiB
CUDA available: True



In [10]:
## 11. Test vLLM Inference
# Run a small model to verify that vLLM can generate text on Tesla T4.
# Use float16 because Tesla T4 does not support bfloat16.

from pathlib import Path
import subprocess

test_script = """
from vllm import LLM, SamplingParams

print("Initializing vLLM...", flush=True)

llm = LLM(
    model="facebook/opt-125m",
    dtype="float16",
    max_model_len=512,
    gpu_memory_utilization=0.6,
    enforce_eager=True,
)

print("Generating text...", flush=True)

outputs = llm.generate(
    ["The best thing about working here is"],
    SamplingParams(
        temperature=0.8,
        max_tokens=40,
    ),
)

print("GENERATED TEXT:", outputs[0].outputs[0].text, flush=True)
"""

Path("/content/test_vllm.py").write_text(test_script)

# Run in the isolated vLLM environment.
with open("/content/vllm_test.log", "w") as log:
    result = subprocess.run(
        ["/content/vllm_env/bin/python", "/content/test_vllm.py"],
        stdout=log,
        stderr=subprocess.STDOUT,
        timeout=300
    )

print("Exit code:", result.returncode)
print(Path("/content/vllm_test.log").read_text()[-5000:])

Exit code: 0
 >= 8
(EngineCore pid=6742) INFO 11-23 11:33:12 [cuda.py:549] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=6742) INFO 11-23 11:33:13 [base_loader.py:95] Initializing model took 6.343466 seconds
(EngineCore pid=6742) INFO 11-23 11:33:18 [weight_utils.py:567] Time spent downloading weights for facebook/opt-125m: 5.271108 seconds

Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]

Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.09it/s]

Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  3.09it/s]

(EngineCore pid=6742) INFO 11-23 11:33:18 [default_loader.py:543] Loading weights took 0.33 seconds
(EngineCore pid=6742) INFO 11-23 11:33:21 [model_runner.py:441] Model loading took 0.24 GiB memory and 15.029486 seconds
(EngineCore pid=6742) WARNING 11-23 11:33:21 [topk_topp_sampler.py:126] FlashInfer top-p/top-k sampling unavailable: FlashInfer kernels are disa

In [11]:
## 11.1. Check GPU Usage
# Display current GPU utilization and memory usage.

!nvidia-smi

Mon Nov 23 11:34:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             28W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
## 11.2. Check vLLM Test Results
# Read the latest messages from the inference test.

from pathlib import Path

log_path = Path("/content/vllm_test.log")

if log_path.exists():
    print(log_path.read_text()[-4000:])
else:
    print("The vLLM test has not created a log file yet.")

 back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=6742) INFO 11-23 11:33:21 [utils.py:419] Using LBNHC KV cache layout.
(EngineCore pid=6742) INFO 11-23 11:33:40 [gpu_worker.py:795] Available KV cache memory: 8.27 GiB
(EngineCore pid=6742) INFO 11-23 11:34:05 [gpu_worker.py:1175] Free memory on device (14.41/14.56 GiB) on startup. Desired GPU memory utilization is (0.6, 8.74 GiB). Actual usage is 0.33 GiB for consumed memory (weights + non-torch), 0.13 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=8723397325` (8.12 GiB) to fit into requested memory, or `--kv-cache-memory=14817930752` (13.8 GiB) to fully utilize gpu memory. Current kv cache memory in use is 8.27 GiB.
(EngineCore pid=6742) INFO 11-23 11:34:05 [extensible_kv_cache.py:693] Memory after warmup: 13.77 GiB free, 0.3 GiB committed to the KV cache, 0.19 GiB kept for transient activations (0.14 GiB allocated at peak, 0.19 GiB of torch r

In [13]:
## 12. Prepare Qwen Model Configuration
# Define the model and conservative GPU settings.
# Do not download or load the model yet.

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

VLLM_CONFIG = {
    "model": MODEL_ID,
    "dtype": "float16",
    "gpu_memory_utilization": 0.85,
    "max_model_len": 2048,
    "max_num_seqs": 1,
    "enforce_eager": True,
}

GENERATION_CONFIG = {
    "temperature": 0.8,
    "top_p": 0.9,
    "max_tokens": 400,
    "seed": 42,
}

print("Model:", MODEL_ID)
print("vLLM configuration:", VLLM_CONFIG)
print("Generation configuration:", GENERATION_CONFIG)

Model: Qwen/Qwen3-4B-Instruct-2507
vLLM configuration: {'model': 'Qwen/Qwen3-4B-Instruct-2507', 'dtype': 'float16', 'gpu_memory_utilization': 0.85, 'max_model_len': 2048, 'max_num_seqs': 1, 'enforce_eager': True}
Generation configuration: {'temperature': 0.8, 'top_p': 0.9, 'max_tokens': 400, 'seed': 42}


In [14]:
## 13. Test Qwen Generation
# Load Qwen3-4B with vLLM and generate one sample review.
# Run the model in the isolated vllm_env environment.

from pathlib import Path
import subprocess

qwen_script = """
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

print("Loading Qwen model...", flush=True)

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
    max_num_seqs=1,
    enforce_eager=True,
)

messages = [
    {
        "role": "system",
        "content": (
            "You write realistic, fictional employee reviews "
            "for a workplace research dataset. "
            "Return exactly three lines: "
            "Headline:, Pros:, and Cons:. "
            "Do not include star ratings or numerical scores."
        ),
    },
    {
        "role": "user",
        "content": (
            "Write one realistic employee review. "
            "The employee appreciates work-life balance, "
            "but is dissatisfied with compensation and senior management. "
            "Make the review natural, specific and balanced."
        ),
    },
]

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.9,
    max_tokens=400,
    seed=42,
)

print("Generating sample review...", flush=True)

outputs = llm.chat(
    messages,
    sampling_params=sampling_params,
)

print("\\nGENERATED REVIEW:", flush=True)
print(outputs[0].outputs[0].text, flush=True)
"""

Path("/content/test_qwen.py").write_text(qwen_script)

with open("/content/qwen_test.log", "w") as log:
    try:
        result = subprocess.run(
            ["/content/vllm_env/bin/python", "/content/test_qwen.py"],
            stdout=log,
            stderr=subprocess.STDOUT,
            timeout=600,
        )
        print("Exit code:", result.returncode)
    except subprocess.TimeoutExpired:
        print("Model loading or generation exceeded the test timeout.")

print(Path("/content/qwen_test.log").read_text()[-6000:])

Exit code: 0
oj.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=8410) WARNING 11-23 11:41:51 [utils.py:345] Attempted to load weight layers.35.post_attention_layernorm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=8410) WARNING 11-23 11:41:51 [utils.py:345] Attempted to load weight norm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16

Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:34<00:00,  9.44s/it]

Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:34<00:00, 11.45s/it]

(EngineCore pid=8410) INFO 11-23 11:41:51 [default_loader.py:543] Loading weights took 34.35 seconds
(EngineCore pid=8410) INFO 11-23 11:41:53 [model_runner.py:441] Model loading took 7.64 GiB memory and 185.284810 seconds
(EngineCore pid=8410) WARNING 11-23 11:41:53 [topk_topp_sampler.py:126] FlashInfer top-p/top-k sampling unavailable: FlashInfer kernels are disabled; falling b

In [15]:
## 14. Load Training Data
# Mount Google Drive and load the prepared training dataset.
# Use only training data for synthetic generation.

from google.colab import drive
from pathlib import Path
import json
import pandas as pd

drive.mount("/content/drive")

DATA_DIR = Path(
    "/content/drive/MyDrive/LLM Project/DATASETS/EXPERIMENTS/v1"
)

train_path = DATA_DIR / "train.jsonl"

assert train_path.exists(), f"File not found: {train_path}"

records = []

with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

train_df = pd.DataFrame(records)

print("Training samples:", len(train_df))
print("Columns:", train_df.columns.tolist())

display(train_df.head(3))

Mounted at /content/drive
Training samples: 20000
Columns: ['id', 'company', 'text', 'work_life_balance', 'compensation_benefits', 'senior_management']


,id,company,text,work_life_balance,compensation_benefits,senior_management
0,177160,Deutsche-Bank,Headline: Mixed Feedback\nPros: If you like ch...,High,High,Low
1,114451,Citi,Headline: Better the devil you know\nPros: It ...,Medium,Low,Low
2,84263,Bloomberg-L-P,Headline: Ailing organisation with bad managem...,Low,Medium,Low


In [16]:
## 15. Prepare C4 Synthetic Generation Samples
# Select target labels and independent style examples.
# Use only training data and keep generation reproducible.

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

LABEL_COLUMNS = [
    "work_life_balance",
    "compensation_benefits",
    "senior_management",
]

# Select five target reviews without replacement.
target_indices = rng.choice(
    len(train_df),
    size=5,
    replace=False
)

pilot_samples = []

for target_idx in target_indices:
    target_row = train_df.iloc[int(target_idx)]

    # Select two different style examples, excluding the target row.
    available_indices = np.delete(
        np.arange(len(train_df)),
        int(target_idx)
    )

    style_indices = rng.choice(
        available_indices,
        size=2,
        replace=False
    )

    style_rows = train_df.iloc[style_indices]

    sample = {
        "source_id": str(target_row["id"]),
        "target_labels": {
            col: target_row[col]
            for col in LABEL_COLUMNS
        },
        "style_examples": style_rows["text"].tolist(),
        "style_source_ids": style_rows["id"].astype(str).tolist(),
    }

    pilot_samples.append(sample)

# Display only the metadata, not the full review texts.
pilot_summary = pd.DataFrame([
    {
        "source_id": sample["source_id"],
        **sample["target_labels"],
        "style_source_ids": sample["style_source_ids"],
    }
    for sample in pilot_samples
])

display(pilot_summary)

print("Pilot samples prepared:", len(pilot_samples))

,source_id,work_life_balance,compensation_benefits,senior_management,style_source_ids
0,338748,High,Low,High,"[115609, 514084]"
1,550606,Medium,Low,Low,"[214474, 103642]"
2,45909,High,Medium,High,"[180120, 472452]"
3,590802,Low,Low,Low,"[15903, 369264]"
4,165544,High,Low,Low,"[144201, 203847]"


Pilot samples prepared: 5


In [17]:
## 16. Generate C4 Pilot Reviews
# Generate five new employee reviews using the selected target labels.
# Use two real reviews as style references only.
# Save the pilot results locally for quality inspection.

import json
import subprocess
from pathlib import Path

PILOT_INPUT = Path("/content/c4_pilot_input.json")
PILOT_OUTPUT = Path("/content/c4_pilot_output.json")
PILOT_SCRIPT = Path("/content/run_c4_pilot.py")
PILOT_LOG = Path("/content/c4_pilot.log")

# Export the five samples from Step 15.
PILOT_INPUT.write_text(
    json.dumps(pilot_samples, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# This script runs inside the isolated vLLM environment.
script = r'''
import json
from pathlib import Path
from vllm import LLM, SamplingParams

input_path = Path("/content/c4_pilot_input.json")
output_path = Path("/content/c4_pilot_output.json")

samples = json.loads(input_path.read_text(encoding="utf-8"))

label_meanings = {
    "Low": "clearly negative or dissatisfied",
    "Medium": "mixed, average, or moderately satisfied",
    "High": "clearly positive or satisfied"
}

def build_messages(sample):
    labels = sample["target_labels"]

    target_description = f"""
Work-life balance: {labels['work_life_balance']}
Meaning: {label_meanings[labels['work_life_balance']]}

Compensation and benefits: {labels['compensation_benefits']}
Meaning: {label_meanings[labels['compensation_benefits']]}

Senior management: {labels['senior_management']}
Meaning: {label_meanings[labels['senior_management']]}
"""

    # Limit style-example length to keep prompts within the context window.
    style_1 = sample["style_examples"][0][:700]
    style_2 = sample["style_examples"][1][:700]

    return [
        {
            "role": "system",
            "content": (
                "You create realistic fictional employee reviews for "
                "a workplace research dataset. "
                "Write naturally, like an actual employee. "
                "Follow the target sentiment for all three workplace aspects. "
                "Do not mention rating categories, scores, or target labels. "
                "Do not copy wording, events, or factual claims from examples. "
                "Do not name any company or real person. "
                "Output exactly three lines beginning with "
                "Headline:, Pros:, and Cons:."
            )
        },
        {
            "role": "user",
            "content": f"""
Write ONE completely new, realistic employee review.

TARGET SENTIMENTS:
{target_description}

STYLE REFERENCES ONLY:
Example 1:
{style_1}

Example 2:
{style_2}

REQUIREMENTS:
- The review must be original.
- Reflect all three target sentiments consistently.
- Use realistic workplace experiences.
- Include meaningful positive and negative observations when appropriate.
- Do not copy content from the examples.
- Keep the complete review between 40 and 1500 characters.
- Do not include company names, star ratings, or numeric scores.
- Return exactly these three lines:

Headline: ...
Pros: ...
Cons: ...
"""
        }
    ]

print("Loading Qwen...", flush=True)

llm = LLM(
    model="Qwen/Qwen3-4B-Instruct-2507",
    dtype="float16",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
    max_num_seqs=1,
    enforce_eager=True
)

sampling = SamplingParams(
    temperature=0.8,
    top_p=0.9,
    max_tokens=400,
    seed=42
)

results = []

for i, sample in enumerate(samples, start=1):
    print(f"Generating review {i}/{len(samples)}...", flush=True)

    output = llm.chat(
        build_messages(sample),
        sampling_params=sampling,
        use_tqdm=False
    )

    generated_text = output[0].outputs[0].text.strip()

    results.append({
        "source_id": sample["source_id"],
        "style_source_ids": sample["style_source_ids"],
        **sample["target_labels"],
        "text": generated_text
    })

    print(generated_text, flush=True)
    print("-" * 60, flush=True)

output_path.write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("PILOT COMPLETE:", len(results), flush=True)
'''

PILOT_SCRIPT.write_text(script, encoding="utf-8")

# Run Qwen in the separate Python environment.
with PILOT_LOG.open("w") as log:
    try:
        result = subprocess.run(
            ["/content/vllm_env/bin/python", str(PILOT_SCRIPT)],
            stdout=log,
            stderr=subprocess.STDOUT,
            timeout=900
        )
        print("Exit code:", result.returncode)
    except subprocess.TimeoutExpired:
        print("Pilot exceeded the timeout.")

# Display the generated reviews or the relevant error.
print(PILOT_LOG.read_text(encoding="utf-8")[-12000:])

Exit code: 0
1778) WARNING 11-23 11:51:33 [utils.py:345] Attempted to load weight layers.32.input_layernorm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=11778) WARNING 11-23 11:51:33 [utils.py:345] Attempted to load weight layers.32.mlp.down_proj.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=11778) WARNING 11-23 11:51:34 [utils.py:345] Attempted to load weight layers.32.post_attention_layernorm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=11778) WARNING 11-23 11:51:34 [utils.py:345] Attempted to load weight layers.32.self_attn.k_norm.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=11778) WARNING 11-23 11:51:34 [utils.py:345] Attempted to load weight layers.32.self_attn.o_proj.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=11778) WARNING 11-23 11:51:34 [utils.py:345] Attempted

In [18]:
## 17. Save C4 Pilot Results
# Save the synthetic pilot to Google Drive for the Midterm.
# These are demonstration samples, not validated training data.

from pathlib import Path
import shutil

pilot_src = Path("/content/c4_pilot_output.json")

pilot_dir = Path(
    "/content/drive/MyDrive/LLM Project/DATASETS/SYNTHETIC/pilot"
)

pilot_dir.mkdir(parents=True, exist_ok=True)

pilot_dst = pilot_dir / "c4_pilot_5_reviews.json"

assert pilot_src.exists(), "Pilot output file not found"

shutil.copy2(pilot_src, pilot_dst)

print("Saved:", pilot_dst)
print("File size:", pilot_dst.stat().st_size, "bytes")

Saved: /content/drive/MyDrive/LLM Project/DATASETS/SYNTHETIC/pilot/c4_pilot_5_reviews.json
File size: 3573 bytes
